# Dhatu Rakshana v4 - Zenodo-first training (coating defects)
**Domain:** the Zenodo CoatingDet images are the target; the wafer dataset is *auxiliary training data only* and is never evaluated.

**Run unattended on Kaggle:** Accelerator GPU T4 x2, Internet ON, Secret `ROBOFLOW_API_KEY` attached, then *Save Version -> Save & Run All (Commit)*.
Optional: *Add Input -> Notebook Output* = your previous run, so its `yolov8_detector_best.pt` is reused as the detector starting point (saves hours).
**Smoke test first:** set `DEBUG_MAX_IMAGES = 300` in the config cell (best effort, ~25 min), then back to `None`.

**What changed vs v3**
* Detector trained on Zenodo with many more background tiles (v3 saw ~1 per clean image; deployment is >95% background) + the detector's own false positives as hard negatives.
* Wafer "particle" removed: it meant *defect* there but *non-critical* in Zenodo, so the two datasets contradicted each other.
* Classifier: 5-fold grouped cross-validation on ALL Zenodo crops (rare classes get real held-out samples + confidence intervals); wafer share capped at 50% per class.
* End-to-end: each image is classified only by the fold model that never saw its specimen group.
* Every metric is on held-out Zenodo data, with an unbiased tile weighting; classes with n<30 are flagged LOW-DATA. No target is assumed met.

**Time budget:** ~4-5 h (download 10 min, tiles 20 min, detector <= 2 h, crops 15 min, classifier CV ~1 h, evaluation ~25 min).

In [1]:
# Step 0: environment
import os, sys, shutil, subprocess, json, time, random, csv, zipfile, urllib.request, math, re, copy
import importlib.metadata as md
from pathlib import Path
from collections import Counter,defaultdict
import numpy as np, torch

CLEAN_WORKING_DIR = False
print(f"Python {sys.version.split()[0]} | PyTorch {torch.__version__}")
cuda_avail = torch.cuda.is_available()
num_gpus = torch.cuda.device_count() if cuda_avail else 0
for i in range(num_gpus):
    p = torch.cuda.get_device_properties(i); print(f"  [GPU {i}] {p.name} | {p.total_memory/1024**3:.1f} GB")
if CLEAN_WORKING_DIR and os.path.exists("/kaggle/working"):
    for item in os.listdir("/kaggle/working"):
        if item != "__notebook__.ipynb":
            fp = os.path.join("/kaggle/working", item)
            shutil.rmtree(fp, ignore_errors=True) if os.path.isdir(fp) else os.remove(fp)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics>=8.2.0", "roboflow", "scikit-learn", "pyyaml", "pandas", "matplotlib"], check=True)
if shutil.which("aria2c") is None:
    subprocess.run(["apt-get", "update", "-qq"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.run(["apt-get", "install", "-y", "-qq", "aria2"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print("Step 0 complete.")

Python 3.13.16 | PyTorch 2.11.0+cu130
  [GPU 0] Tesla T4 | 14.6 GB
  [GPU 1] Tesla T4 | 14.6 GB


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


Step 0 complete.


In [2]:
# Step 1: configuration (Zenodo-first)
from PIL import Image
SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_everything()

IS_KAGGLE = Path("/kaggle/working").exists()
WORKING_DIR = Path("/kaggle/working") if IS_KAGGLE else Path("./kaggle_output")
TEMP_DIR    = Path("/tmp") if IS_KAGGLE else Path("./kaggle_temp")
INPUT_DIR   = Path("/kaggle/input") if IS_KAGGLE else Path("./kaggle_input")
WORKING_DIR.mkdir(parents=True, exist_ok=True); TEMP_DIR.mkdir(parents=True, exist_ok=True)

TILES_DIR            = TEMP_DIR / "tiles_v4"
CROPS_DIR            = TEMP_DIR / "crops_v4"
ROBOFLOW_DIR         = TEMP_DIR / "roboflow_dataset"
ZENODO_EXTRACTED_DIR = TEMP_DIR / "coatingdet_extracted"
ZENODO_ZIP_PATH      = TEMP_DIR / "CoatingDet_v2.zip"
ZENODO_DOWNLOAD_URL  = "https://zenodo.org/api/records/21600391/files/CoatingDet_v2.zip/content"
ZENODO_MIN_BYTES     = 13 * 1024**3
for d in (TILES_DIR, CROPS_DIR): shutil.rmtree(d, ignore_errors=True)

MAX_TEST3_IMAGES = None
DEBUG_MAX_IMAGES = None          # e.g. 300 = smoke test of the whole pipeline (~25 min). Set back to None for the real run.
VAL_FRAC, TEST_FRAC = 0.15, 0.15 # detector split (grouped by specimen/session, Zenodo only)
SPLITS = ("train", "val", "test")
K_FOLDS = 5                      # classifier: grouped cross-validation over ALL Zenodo crops

# ---- label semantics ----
ZENODO_DEFECT_CIDS, ZENODO_PARTICLE_CIDS = {0}, {1}          # 1 = minor particle (non-critical) -> detector class 1, classifier no_defect
DET_NAMES = {0: "defect", 1: "particle"}
STAGE2_CLASSES = ["contamination", "inclusion", "no_defect", "pinhole", "scratch"]
c2i = {c: i for i, c in enumerate(STAGE2_CLASSES)}; i2c = {i: c for c, i in c2i.items()}
DEFECT_CLASSES = [c for c in STAGE2_CLASSES if c != "no_defect"]

# ---- wafer data = AUXILIARY ONLY (training, never evaluated) ----
USE_WAFER_AUX = True
# Wafer "particle" is deliberately NOT mapped: in the wafer set it is a defect (-> inclusion), in Zenodo "particle" is NON-critical (-> no_defect).
# Images containing any unmapped wafer label are dropped so the two datasets never teach contradictory things.
WAFER_LABEL_MAP = {"crack": "scratch", "cracks": "scratch", "scratch": "scratch", "scratches": "scratch",
                   "pinhole": "pinhole", "pinholes": "pinhole", "speckle": "contamination", "speckles": "contamination", "contamination": "contamination"}
WAFER_TILE_CAP, WAFER_TILES_PER_IMG, WAFER_BG_RATIO = 4000, 2, 0.15
WAFER_CROP_CAP_PER_CLASS = 1500
RF_BLOCK = 10

# ---- Stage 1: detector ----
DET_INIT_WEIGHTS = "yolov8m.pt"
REUSE_PREVIOUS_DETECTOR = True   # if a previous yolov8_detector_best.pt is in /kaggle/working or an attached input, start from it (saves hours)
DET_IMGSZ = 640
DET_A_HOURS, DET_A_EPOCHS = 1.5, 40          # stage A: background-heavy training (hard wall-clock cap)
DET_B_HOURS, DET_B_EPOCHS = 0.5, 15          # stage B: + mined false-positive tiles
TILE_SIZE, TILE_OVERLAP, TILE_MIN_VISIBLE = 512, 128, 0.25
BG_PER_LABELLED, BG_PER_CLEAN_IMG = 3, 4     # Step 4 of v3 used ~0.15 / 1: far too few backgrounds for images that are >95% background
HN_IMAGES, HN_TILES_PER_IMG, HN_CONF, HN_CAP = 1200, 12, 0.15, 3000
DET_BATCH = 32 if num_gpus >= 2 else 16
DET_DEVICES = [0, 1] if num_gpus >= 2 else (0 if num_gpus == 1 else "cpu")
PRED_DEVICE = 0 if cuda_avail else "cpu"
NMS_GRID, CONF_GRID = [0.3, 0.4, 0.5, 0.6, 0.7], [0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]

# ---- Stage 2: classifier ----
CLS_BACKBONE = "resnet34"
import torchvision.models as _tvm
assert hasattr(_tvm, CLS_BACKBONE), f"CLS_BACKBONE={CLS_BACKBONE!r} is not a torchvision model"
CLS_IMG_SIZE, CLS_BATCH = 256, 64
CLS_WARMUP_EPOCHS, CLS_EPOCHS, CLS_PATIENCE = 2, 25, 6
EPOCH_CAP = 800                  # samples per class per epoch (rare Zenodo classes are oversampled; wafer share capped at 50%)
LABEL_SMOOTH, EMA_DECAY = 0.05, 0.995
PARTICLE_PER_IMAGE, PARTICLE_CROP_CAP, BG_CROP_CAP = 3, 3000, 2500
HN_CROP_IMAGES, HN_CROP_CAP = 1500, 1500

# ---- end-to-end evaluation ----
E2E_DEF_IMGS, E2E_CLEAN_IMGS = 100, 100
AP_EVAL_DEF_IMGS, AP_EVAL_CLEAN_IMGS = 100, 100

print(f"Detector {DET_INIT_WEIGHTS} @ {DET_IMGSZ} | stage A <= {DET_A_HOURS} h, stage B <= {DET_B_HOURS} h | batch {DET_BATCH}")
print(f"Classifier {CLS_BACKBONE} @ {CLS_IMG_SIZE} | {K_FOLDS}-fold grouped CV | wafer aux: {USE_WAFER_AUX}")

Detector yolov8m.pt @ 640 | stage A <= 1.5 h, stage B <= 0.5 h | batch 32
Classifier resnet34 @ 256 | 5-fold grouped CV | wafer aux: True


In [3]:
# Step 1a: shared geometry / split helpers (reused unchanged from v3)
# ---- shared geometry / split helpers ----
def slice_grid(W, H, tile=TILE_SIZE, overlap=TILE_OVERLAP):
    stride = tile - overlap
    def axis(n):
        if n <= tile: return [0]
        pos = list(range(0, n - tile + 1, stride))
        if pos[-1] + tile < n: pos.append(n - tile)
        return pos
    return [(x, y) for y in axis(H) for x in axis(W)]

def crop_side(bw, bh, W, H, min_side=48):
    m = max(bw, bh); return min(max(m + 2 * (m * 0.35 + 16), min_side), min(W, H))

def context_crop(im, box, min_side=48):
    W, H = im.size
    x0, y0, x1, y1 = [float(v) for v in box]
    side = crop_side(x1 - x0, y1 - y0, W, H, min_side)
    cx, cy = (x0 + x1) / 2.0, (y0 + y1) / 2.0
    left = min(max(cx - side / 2.0, 0), W - side); top = min(max(cy - side / 2.0, 0), H - side)
    return im.crop((int(left), int(top), int(left + side), int(top + side)))

def iou_xyxy(a, b):
    ix0, iy0, ix1, iy1 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, ix1 - ix0) * max(0.0, iy1 - iy0)
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / ua if ua > 0 else 0.0

def source_id(stem): return stem.split(".rf.")[0]

_ZEN_A = re.compile(r"^(\d{8})T\d{6}Z_\d+_(\d+)_([A-Za-z]+)_raw")
_ZEN_B = re.compile(r"^(\d{4}-\d{2}-\d{2})__(\d{2})-(\d{2})-\d{2}")
_RF_NUM = re.compile(r"^(\d+)_bmp")
def group_key(stem, domain):
    """Frames of the same specimen / session / consecutive wafer ids must never straddle splits.
    ASSUMPTION: 3rd numeric token of Zenodo names is a specimen/sample id. Check the group counts printed in Step 3."""
    if domain == "zenodo":
        m = _ZEN_A.match(stem)
        if m: return f"z_{m.group(1)}_{m.group(2)}_{m.group(3)}"
        m = _ZEN_B.match(stem)
        if m: return f"z_{m.group(1)}_{m.group(2)}_{int(m.group(3)) // 10}"
        return "z_" + stem
    s = source_id(stem); m = _RF_NUM.match(s)
    return f"rf_blk{int(m.group(1)) // RF_BLOCK}" if m else "rf_" + s

def assign_splits(items, val_frac=VAL_FRAC, test_frac=TEST_FRAC, seed=SEED):
    """items: (group, label). Group-level, stratified by the RAREST label found in the group."""
    rng = random.Random(seed)
    labs = defaultdict(set); freq = Counter()
    for g, lab in items:
        if lab not in labs[g]: freq[lab] += 1
        labs[g].add(lab)
    strat = {}
    for g, s in labs.items():
        non_clean = [l for l in s if l != "clean"]
        strat[g] = min(non_clean or ["clean"], key=lambda l: (freq[l], l))
    by_label = defaultdict(list)
    for g, l in strat.items(): by_label[l].append(g)
    out = {}
    for lab in sorted(by_label):
        gs = sorted(by_label[lab]); rng.shuffle(gs); n = len(gs)
        n_test, n_val = int(round(n * test_frac)), int(round(n * val_frac))
        if n >= 3: n_test, n_val = max(n_test, 1), max(n_val, 1)
        for i, g in enumerate(gs):
            out[g] = "test" if i < n_test else ("val" if i < n_test + n_val else "train")
    return out

In [4]:
# Step 1b: extra helpers (tiles, folds, mining)
def tile_gt(boxes, tx, ty):
    """Labelled boxes of one tile in tile coordinates: (defects, particles); a box counts if >= TILE_MIN_VISIBLE of it is inside."""
    d, p = [], []
    for role, _, bx0, by0, bx1, by1 in boxes:
        ix0, iy0, ix1, iy1 = max(tx, bx0), max(ty, by0), min(tx + TILE_SIZE, bx1), min(ty + TILE_SIZE, by1)
        if ix1 > ix0 and iy1 > iy0:
            frac = ((ix1 - ix0) * (iy1 - iy0)) / max((bx1 - bx0) * (by1 - by0), 1.0)
            if frac >= TILE_MIN_VISIBLE: (d if role == "defect" else p).append((ix0 - tx, iy0 - ty, ix1 - tx, iy1 - ty))
    return d, p

def touches_any(boxes, tx, ty):
    """True if ANY labelled box overlaps the tile at all -> never used as clean background."""
    for _, _, x0, y0, x1, y1 in boxes:
        if min(tx + TILE_SIZE, x1) > max(tx, x0) and min(ty + TILE_SIZE, y1) > max(ty, y0): return True
    return False

def yolo_lines(d, p):
    out = []
    for cls, bxs in ((0, d), (1, p)):
        for x0, y0, x1, y1 in bxs:
            out.append(f"{cls} {(x0+x1)/2/TILE_SIZE:.6f} {(y0+y1)/2/TILE_SIZE:.6f} {(x1-x0)/TILE_SIZE:.6f} {(y1-y0)/TILE_SIZE:.6f}")
    return out

def write_tile(im, out_dir, stem, tx, ty, d, p):
    out_dir = Path(out_dir); (out_dir / "images").mkdir(parents=True, exist_ok=True); (out_dir / "labels").mkdir(parents=True, exist_ok=True)
    name = f"{stem}_{tx}_{ty}"
    if (out_dir / "images" / f"{name}.jpg").exists(): return False
    im.crop((tx, ty, tx + TILE_SIZE, ty + TILE_SIZE)).save(out_dir / "images" / f"{name}.jpg", quality=95)
    (out_dir / "labels" / f"{name}.txt").write_text("\n".join(yolo_lines(d, p)))
    return True

def assign_folds(items, k=K_FOLDS, seed=SEED):
    """items: (group, label). Group-level folds, stratified by the RAREST non-clean label of each group, rotating offsets so folds stay balanced."""
    rng = random.Random(seed); labs = defaultdict(set); freq = Counter()
    for g, lab in items:
        if lab not in labs[g]: freq[lab] += 1
        labs[g].add(lab)
    strat = {}
    for g, s in labs.items():
        non_clean = [l for l in s if l != "clean"]
        strat[g] = min(non_clean or ["clean"], key=lambda l: (freq[l], l))
    by_label = defaultdict(list)
    for g, l in strat.items(): by_label[l].append(g)
    out, off = {}, 0
    for lab in sorted(by_label):
        gs = sorted(by_label[lab]); rng.shuffle(gs)
        for i, g in enumerate(gs): out[g] = (i + off) % k
        off += len(gs)
    return out

def mine_fp_tiles(det, recs, n_img, tiles_per_img, conf, seed):
    """Yield (rec, tx, ty, tile_image, boxes_xyxy) for background tiles on which the detector fires (= false positives)."""
    rng = random.Random(seed)
    for rec in rng.sample(recs, min(n_img, len(recs))):
        with Image.open(rec["path"]) as im0: im = im0.convert("RGB")
        W, H = im.size
        cand = [(tx, ty) for tx, ty in slice_grid(W, H) if not touches_any(rec["boxes"], tx, ty)]
        pick = rng.sample(cand, min(tiles_per_img, len(cand)))
        for s in range(0, len(pick), 32):
            chunk = pick[s:s+32]; tiles = [im.crop((tx, ty, tx + TILE_SIZE, ty + TILE_SIZE)) for tx, ty in chunk]
            res = det.predict(tiles, imgsz=DET_IMGSZ, conf=conf, iou=0.5, classes=[0], verbose=False, device=PRED_DEVICE)
            for (tx, ty), t_img, r in zip(chunk, tiles, res):
                if len(r.boxes): yield rec, tx, ty, t_img, r.boxes.xyxy.cpu().numpy()

In [5]:
# Step 2: dataset ingestion (unchanged logic)
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}
def list_images(d): return sorted(p for p in Path(d).iterdir() if p.suffix.lower() in IMG_EXTS)

def selective_unpack_zenodo(zip_path, extract_root, max_test3_images=None):
    extract_root.mkdir(parents=True, exist_ok=True)
    meta_members, train_members, t3_imgs, t3_lbls = [], [], [], {}
    with zipfile.ZipFile(zip_path, "r") as zf:
        for m in zf.infolist():
            norm = m.filename.replace("\\", "/"); rel = norm
            if "/" in norm and not norm.startswith(("train/", "test_3/", "metadata", "class_labels")):
                rel = norm.split("/", 1)[1]
            if rel.startswith("metadata.csv") or rel.startswith("class_labels.yaml"): meta_members.append(m)
            elif rel.startswith(("train/raw_data/images/", "train/raw_data/labels/")):
                if not m.is_dir(): train_members.append(m)
            elif rel.startswith("test_3/raw_data/images/"):
                if not m.is_dir() and rel.lower().endswith(tuple(IMG_EXTS)): t3_imgs.append((rel, m))
            elif rel.startswith("test_3/raw_data/labels/"):
                if not m.is_dir() and rel.lower().endswith(".txt"): t3_lbls[Path(rel).stem] = m
        t3_imgs.sort(key=lambda x: x[0])
        sel = [m for _, m in (t3_imgs if max_test3_images is None else t3_imgs[:max_test3_images])]
        stems = {Path(m.filename.replace("\\", "/")).stem for m in sel}
        selected = meta_members + train_members + sel + [t3_lbls[s] for s in stems if s in t3_lbls]
        print(f"Unpacking {len(selected)} items ...")
        for i, m in enumerate(selected):
            zf.extract(m, extract_root)
            if (i + 1) % 3000 == 0: print(f"  {i+1}/{len(selected)}")

import shutil, subprocess, zipfile, urllib.request

def _zip_ok(p):
    """True only if the file really is a complete zip (central directory readable)."""
    try:
        if not (p.exists() and zipfile.is_zipfile(p)): return False
        with zipfile.ZipFile(p) as zf: return len(zf.namelist()) > 0
    except Exception:
        return False

def _purge(p):
    for f in (p, Path(str(p) + ".aria2")):
        f.unlink(missing_ok=True)

def download_zenodo(max_tries=3):
    free_gb = shutil.disk_usage(ZENODO_ZIP_PATH.parent).free / 1024**3
    print(f"Free disk in {ZENODO_ZIP_PATH.parent}: {free_gb:.1f} GB")
    if _zip_ok(ZENODO_ZIP_PATH) and ZENODO_ZIP_PATH.stat().st_size >= ZENODO_MIN_BYTES:
        print("Existing zip is valid, skipping download."); return
    if ZENODO_ZIP_PATH.exists():
        print("Existing zip is incomplete/corrupt -> deleting and re-downloading.")
        _purge(ZENODO_ZIP_PATH)
    if free_gb < 16:
        raise RuntimeError(f"Only {free_gb:.1f} GB free; need ~16 GB for the zip plus extraction. "
                           "Restart the session / free space, or attach the dataset as a Kaggle input instead.")
    for attempt in range(1, max_tries + 1):
        print(f"Downloading CoatingDet_v2.zip (attempt {attempt}/{max_tries}) ...")
        if shutil.which("aria2c"):
            subprocess.run(["aria2c", "-x8", "-s8", "-k1M", "-c", "--file-allocation=none", "--max-tries=10", "--retry-wait=5",
                            "--summary-interval=0", "--console-log-level=error",
                            "-d", str(ZENODO_ZIP_PATH.parent), "-o", ZENODO_ZIP_PATH.name, ZENODO_DOWNLOAD_URL])
        else:
            with urllib.request.urlopen(ZENODO_DOWNLOAD_URL) as resp, open(ZENODO_ZIP_PATH, "wb") as f:
                shutil.copyfileobj(resp, f)
        size = ZENODO_ZIP_PATH.stat().st_size if ZENODO_ZIP_PATH.exists() else 0
        print(f"  downloaded {size/1024**3:.2f} GB")
        if size >= ZENODO_MIN_BYTES and _zip_ok(ZENODO_ZIP_PATH):
            print("  zip verified."); return
        print("  not a valid zip yet.")
        if size < ZENODO_MIN_BYTES:
            continue                      # partial: keep it, aria2 -c will resume
        _purge(ZENODO_ZIP_PATH)           # full size but unreadable: start over
    raise RuntimeError("Download failed after retries. Check the network, or attach the dataset as a Kaggle input.")

attached = IS_KAGGLE and any(INPUT_DIR.glob("**/raw_data/images"))
extracted_ready = ZENODO_EXTRACTED_DIR.exists() and any(ZENODO_EXTRACTED_DIR.rglob("raw_data/images"))
if attached: print("Using attached CoatingDet dataset")
elif not extracted_ready:
    download_zenodo(); selective_unpack_zenodo(ZENODO_ZIP_PATH, ZENODO_EXTRACTED_DIR, MAX_TEST3_IMAGES); ZENODO_ZIP_PATH.unlink(missing_ok=True)

search_roots = [r for r in [ZENODO_EXTRACTED_DIR] + ([INPUT_DIR] if IS_KAGGLE else []) if r.exists()]
train_dirs = sorted({d for r in search_roots for d in r.rglob("train/raw_data/images")})
test3_dirs = sorted({d for r in search_roots for d in r.rglob("test_3/raw_data/images")})
zenodo_imgs = sorted({p for d in train_dirs + test3_dirs for p in list_images(d)})
assert zenodo_imgs, "0 Zenodo images found"
if DEBUG_MAX_IMAGES:
    zenodo_imgs = sorted(random.Random(SEED).sample(zenodo_imgs, min(DEBUG_MAX_IMAGES, len(zenodo_imgs))))
print("Zenodo images:", len(zenodo_imgs))

def parse_metadata_csv(meta_path):
    mapping = {}
    with open(meta_path, "r", encoding="utf-8-sig") as f:
        sample = f.read(4096); f.seek(0)
        for row in csv.DictReader(f, delimiter=";" if ";" in sample else ","):
            fn = row.get("filename") or row.get("file_name") or row.get("image")
            lbl = row.get("label") or row.get("subtype") or row.get("class")
            if fn and lbl: mapping[Path(fn).stem] = lbl.lower().strip()
    return mapping
meta_cands = sorted({p for r in search_roots for p in r.rglob("metadata.csv")})
meta_dict = parse_metadata_csv(meta_cands[0]) if meta_cands else {}
print("metadata labels:", dict(Counter(meta_dict.values())))

Free disk in /tmp: 2514.5 GB
[#6eabc7 224KiB/14GiB(0%) CN:8 DL:519KiB ETA:7h56m30s]
[#6eabc7 5.4MiB/14GiB(0%) CN:8 DL:3.5MiB ETA:1h8m57s]
[#6eabc7 11MiB/14GiB(0%) CN:8 DL:4.5MiB ETA:53m17s]
[#6eabc7 18MiB/14GiB(0%) CN:8 DL:5.2MiB ETA:45m46s]
[#6eabc7 25MiB/14GiB(0%) CN:8 DL:5.6MiB ETA:42m45s]
[#6eabc7 32MiB/14GiB(0%) CN:8 DL:5.8MiB ETA:41m16s]
[#6eabc7 39MiB/14GiB(0%) CN:8 DL:6.0MiB ETA:40m9s]
[#6eabc7 47MiB/14GiB(0%) CN:8 DL:6.2MiB ETA:38m49s]
[#6eabc7 76MiB/14GiB(0%) CN:8 DL:8.8MiB ETA:27m4s]
[#6eabc7 96MiB/14GiB(0%) CN:8 DL:9.9MiB ETA:24m8s]
[#6eabc7 108MiB/14GiB(0%) CN:8 DL:11MiB ETA:21m40s]
[#6eabc7 121MiB/14GiB(0%) CN:8 DL:11MiB ETA:20m20s]
[#6eabc7 133MiB/14GiB(0%) CN:8 DL:12MiB ETA:19m33s]
[#6eabc7 144MiB/14GiB(0%) CN:8 DL:12MiB ETA:18m50s]
[#6eabc7 167MiB/14GiB(1%) CN:8 DL:14MiB ETA:16m36s]
[#6eabc7 190MiB/14GiB(1%) CN:8 DL:15MiB ETA:14m55s]
[#6eabc7 203MiB/14GiB(1%) CN:8 DL:16MiB ETA:14m18s]
[#6eabc7 235MiB/14GiB(1%) CN:8 DL:19MiB ETA:12m28s]
[#6eabc7 261MiB/14GiB(1%) CN:8 DL

In [6]:
# Step 3: manifest. Zenodo = the evaluated domain (grouped split). Wafer = auxiliary training data only.
import yaml
def parse_yolo_boxes(label_path, W, H):
    boxes = []
    if not label_path or not os.path.exists(label_path): return boxes
    for line in open(label_path, "r", encoding="utf-8"):
        parts = line.strip().split()
        if len(parts) >= 5:
            try: cid = int(float(parts[0])); cx, cy, bw, bh = map(float, parts[1:5])
            except ValueError: continue
            boxes.append((cid, max(0, (cx-bw/2)*W), max(0, (cy-bh/2)*H), min(W, (cx+bw/2)*W), min(H, (cy+bh/2)*H)))
    return boxes

def find_label(img_p):
    for cand in (img_p.parent.parent / "labels" / f"{img_p.stem}.txt", img_p.parent.parent / "raw_data" / "labels" / f"{img_p.stem}.txt", img_p.parent / f"{img_p.stem}.txt"):
        if cand.exists(): return cand
    return None

def resolve_subtype(stem):
    raw = meta_dict.get(stem)
    return raw if raw in DEFECT_CLASSES else None

zen_records, audit, cid_hist = [], Counter(), Counter()
for p in zenodo_imgs:
    try:
        with Image.open(p) as im: W, H = im.size
    except Exception: audit["zenodo_unreadable"] += 1; continue
    subtype = resolve_subtype(p.stem); boxes = []
    for cid, x0, y0, x1, y1 in parse_yolo_boxes(find_label(p), W, H):
        cid_hist[cid] += 1
        if cid in ZENODO_DEFECT_CIDS: boxes.append(("defect", subtype, x0, y0, x1, y1))
        elif cid in ZENODO_PARTICLE_CIDS: boxes.append(("particle", None, x0, y0, x1, y1))
        else: audit["zenodo_unknown_cid"] += 1
    n_def = sum(b[0] == "defect" for b in boxes)
    if n_def and subtype is None: audit["zenodo_defect_images_without_subtype"] += 1
    strat = (subtype or "unresolved") if n_def else "clean"
    zen_records.append(dict(path=p, stem=p.stem, group=group_key(p.stem, "zenodo"), domain="zenodo", W=W, H=H, boxes=boxes, strat=strat))
print("Zenodo class-id histogram:", dict(cid_hist), "| audit:", dict(audit))

split_of = assign_splits([(r["group"], r["strat"]) for r in zen_records])
for r in zen_records: r["split"] = split_of[r["group"]]
train_recs = [r for r in zen_records if r["split"] == "train"]
val_recs   = [r for r in zen_records if r["split"] == "val"]
test_recs  = [r for r in zen_records if r["split"] == "test"]
print(f"Zenodo: {len(zen_records)} images in {len({r['group'] for r in zen_records})} groups -> train {len(train_recs)} / val {len(val_recs)} / test {len(test_recs)}")
tab = Counter((b[1] or "unresolved", r["split"]) for r in zen_records for b in r["boxes"] if b[0] == "defect")
print("Zenodo defect boxes by subtype/split (small cells = noisy metrics):", dict(sorted(tab.items())))

# --- wafer auxiliary data ---
wafer_records, wafer_dropped = [], Counter()
if USE_WAFER_AUX:
    try:
        from kaggle_secrets import UserSecretsClient
        ROBOFLOW_API_KEY = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
    except Exception: ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY", "")
    rf_dir = None
    rf_cands = sorted(INPUT_DIR.glob("**/*thin-film*")) if IS_KAGGLE else []
    if rf_cands: rf_dir = rf_cands[0]
    elif ROBOFLOW_DIR.exists() and any(ROBOFLOW_DIR.rglob("data.yaml")): rf_dir = ROBOFLOW_DIR
    elif ROBOFLOW_API_KEY:
        try:
            from roboflow import Roboflow
            Roboflow(api_key=ROBOFLOW_API_KEY).workspace("thin-film-defect-detection").project("defect-detection-kxsyt").version(43).download("yolov8", location=str(ROBOFLOW_DIR))
            rf_dir = ROBOFLOW_DIR
        except Exception as e: print("Roboflow download failed:", e)
    print("Wafer (Roboflow) dataset:", rf_dir)
    if rf_dir is None:
        print("WARNING: wafer data not available -> continuing with Zenodo only (attach secret ROBOFLOW_API_KEY or set USE_WAFER_AUX=False to silence).")
    else:
        rf_names = {0: "crack", 1: "particle", 2: "pinhole", 3: "speckle"}
        ycs = sorted(Path(rf_dir).rglob("data.yaml"))
        if ycs:
            names = yaml.safe_load(open(ycs[0], encoding="utf-8")).get("names", {})
            rf_names = {i: str(n).lower().strip() for i, n in (enumerate(names) if isinstance(names, list) else names.items())}
        print("Wafer class names:", rf_names)
        rf_imgs = sorted(list(Path(rf_dir).rglob("*.jpg")) + list(Path(rf_dir).rglob("*.png")))
        if DEBUG_MAX_IMAGES: rf_imgs = random.Random(SEED).sample(rf_imgs, min(DEBUG_MAX_IMAGES, len(rf_imgs)))
        for p in rf_imgs:
            lbl_p = p.parent.parent / "labels" / f"{p.stem}.txt"
            if not lbl_p.exists(): lbl_p = p.with_suffix(".txt")
            if not lbl_p.exists(): continue
            try:
                with Image.open(p) as im: W, H = im.size
            except Exception: continue
            raw_boxes = parse_yolo_boxes(lbl_p, W, H); boxes, ok = [], bool(raw_boxes)
            for cid, x0, y0, x1, y1 in raw_boxes:
                lab = WAFER_LABEL_MAP.get(rf_names.get(int(cid), ""))
                if lab is None: ok = False; wafer_dropped[rf_names.get(int(cid), str(cid))] += 1; break
                boxes.append(("defect", lab, x0, y0, x1, y1))
            if ok: wafer_records.append(dict(path=p, stem=p.stem, group=group_key(p.stem, "roboflow"), domain="wafer", W=W, H=H, boxes=boxes, strat="aux", split="aux"))
        print(f"Wafer aux images kept: {len(wafer_records)} | images dropped because of an unmapped class (e.g. particle): {dict(wafer_dropped)}")
records = zen_records + wafer_records

Zenodo class-id histogram: {1: 1517, 0: 668} | audit: {}
Zenodo: 4929 images in 1248 groups -> train 3393 / val 623 / test 913
Zenodo defect boxes by subtype/split (small cells = noisy metrics): {('contamination', 'test'): 6, ('contamination', 'train'): 33, ('contamination', 'val'): 5, ('inclusion', 'test'): 88, ('inclusion', 'train'): 357, ('inclusion', 'val'): 76, ('pinhole', 'test'): 3, ('pinhole', 'train'): 13, ('pinhole', 'val'): 3, ('scratch', 'test'): 28, ('scratch', 'train'): 49, ('scratch', 'val'): 7}
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /tmp/roboflow_dataset in yolov8:: 100%|██████████| 12318/12318 [00:01<00:00, 7603.99it/s]


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Wafer (Roboflow) dataset: /tmp/roboflow_dataset
Wafer class names: {0: 'crack', 1: 'particle', 2: 'pinhole', 3: 'speckle'}
Wafer aux images kept: 954 | images dropped because of an unmapped class (e.g. particle): {'particle': 5161}


In [7]:
# Step 4: detector tiles. Zenodo: every labelled tile + MANY background tiles; wafer: capped auxiliary tiles. No test tiles are written.
def write_zenodo_tiles(rec, split):
    out = TILES_DIR / "zenodo" / split; rng = random.Random(f"{SEED}-T-{rec['stem']}")
    with Image.open(rec["path"]) as im0: im = im0.convert("RGB")
    W, H = im.size; lab, bgc = [], []
    for tx, ty in slice_grid(W, H):
        d, p = tile_gt(rec["boxes"], tx, ty)
        if d or p: lab.append((tx, ty, d, p))
        elif not touches_any(rec["boxes"], tx, ty): bgc.append((tx, ty))
    n_l = sum(write_tile(im, out, rec["stem"], tx, ty, d, p) for tx, ty, d, p in lab)
    n_bg_target = BG_PER_LABELLED * len(lab) if lab else BG_PER_CLEAN_IMG
    n_b = sum(write_tile(im, out, rec["stem"], tx, ty, [], []) for tx, ty in rng.sample(bgc, min(n_bg_target, len(bgc))))
    return n_l, n_b

t0 = time.time(); tile_stats = Counter()
for sp, recs in (("train", train_recs), ("val", val_recs)):
    for i, rec in enumerate(recs):
        l, b = write_zenodo_tiles(rec, sp); tile_stats[f"zenodo/{sp}/labelled"] += l; tile_stats[f"zenodo/{sp}/background"] += b
        if (i + 1) % 500 == 0: print(f"  zenodo {sp} {i+1}/{len(recs)} ({time.time()-t0:.0f}s)", flush=True)

wafer_tiles_written = 0
if wafer_records:
    wr = list(wafer_records); random.Random(SEED).shuffle(wr)
    for rec in wr:
        if wafer_tiles_written >= WAFER_TILE_CAP: break
        rng = random.Random(f"{SEED}-W-{rec['stem']}"); out = TILES_DIR / "wafer" / "train"
        with Image.open(rec["path"]) as im0: im = im0.convert("RGB")
        W, H = im.size; lab, bgc = [], []
        for tx, ty in slice_grid(W, H):
            d, p = tile_gt(rec["boxes"], tx, ty)
            if d: lab.append((tx, ty, d, p))
            elif not touches_any(rec["boxes"], tx, ty): bgc.append((tx, ty))
        lab = rng.sample(lab, min(WAFER_TILES_PER_IMG, len(lab)))
        for tx, ty, d, p in lab: wafer_tiles_written += write_tile(im, out, rec["stem"], tx, ty, d, p)
        for tx, ty in rng.sample(bgc, min(max(1, math.ceil(WAFER_BG_RATIO * len(lab))) if lab else 0, len(bgc))): write_tile(im, out, rec["stem"], tx, ty, [], [])
tile_stats["wafer/train/labelled"] = wafer_tiles_written
print("Tiles written:", dict(tile_stats), f"| {time.time()-t0:.0f}s")

def write_det_yaml(path, train_dirs):
    Path(path).write_text(yaml.safe_dump({"path": str(TILES_DIR.resolve()), "train": train_dirs, "val": ["zenodo/val/images"], "names": DET_NAMES}))
train_dirs = ["zenodo/train/images"] + (["wafer/train/images"] if wafer_tiles_written else [])
write_det_yaml(TILES_DIR / "data.yaml", train_dirs)

  zenodo train 500/3393 (88s)
  zenodo train 1000/3393 (172s)
  zenodo train 1500/3393 (255s)
  zenodo train 2000/3393 (338s)
  zenodo train 2500/3393 (423s)
  zenodo train 3000/3393 (515s)
  zenodo val 500/623 (672s)
Tiles written: {'zenodo/train/labelled': 2853, 'zenodo/train/background': 18107, 'zenodo/val/labelled': 526, 'zenodo/val/background': 3266, 'wafer/train/labelled': 1868} | 706s


In [8]:
# Step 5: Stage 1A - detector training (hard time cap)
from ultralytics import YOLO
init = DET_INIT_WEIGHTS; reused = False
if REUSE_PREVIOUS_DETECTOR:
    for root in (WORKING_DIR, INPUT_DIR):
        found = sorted(root.glob("**/yolov8_detector_best.pt")) if root.exists() else []
        if found: init, reused = str(found[0]), True; break
print(f"--- Stage 1A: init from {init} ({'previous detector' if reused else 'COCO pretrained'}) | imgsz {DET_IMGSZ} | cap {DET_A_HOURS} h ---")
t0 = time.time()
mA = YOLO(init)
mA.train(data=str((TILES_DIR / "data.yaml").resolve()), epochs=DET_A_EPOCHS, time=DET_A_HOURS, batch=DET_BATCH, imgsz=DET_IMGSZ, device=DET_DEVICES,
         project=str(TEMP_DIR / "runs_v4"), name="detector_A", exist_ok=True, workers=4, seed=SEED, deterministic=False,
         optimizer="AdamW", lr0=3e-4 if reused else 1e-3, lrf=0.05, cos_lr=True, warmup_epochs=1.0 if reused else 3.0,
         mosaic=0.5, close_mosaic=8, scale=0.3, flipud=0.5, fliplr=0.5, degrees=0.0, mixup=0.0, erasing=0.0,
         amp=True, patience=12, save=True, plots=False, verbose=False)
det_A_path = Path(mA.trainer.save_dir) / "weights" / "best.pt"
print(f"Stage 1A done in {(time.time()-t0)/60:.1f} min -> {det_A_path}")

--- Stage 1A: init from yolov8m.pt (COCO pretrained) | imgsz 640 | cap 1.5 h ---
Ultralytics 8.4.218 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=8, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/tmp/tiles_v4/data.yaml, degrees=0.0, deterministic=False, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.0, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.001, lrf=0.05, mask_ratio=4, ma

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/16         7G      1.859      2.484      1.854          2        640: 100% ━━━━━━━━━━━━ 740/740 2.3it/s 5:28
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 60/60 2.8it/s 21.3s
                   all       3792        546      0.184      0.467       0.27      0.125

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/16         7G       1.83      2.377      1.844          1        640: 100% ━━━━━━━━━━━━ 740/740 2.3it/s 5:24
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 60/60 2.8it/s 21.2s
                   all       3792        546      0.297      0.398       0.25      0.109

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/16         7G      1.756      2.312      1.769          0        640: 100% ━━━━━━━━━

In [9]:
# Step 6: Stage 1B - mine the detector's own false positives on TRAIN images and fine-tune on them
det_A = YOLO(str(det_A_path)); n_hn = 0; t0 = time.time()
for rec, tx, ty, t_img, bxs in mine_fp_tiles(det_A, train_recs, HN_IMAGES, HN_TILES_PER_IMG, HN_CONF, SEED):
    if n_hn >= HN_CAP: break
    n_hn += write_tile(t_img, TILES_DIR / "zenodo" / "train", f"{rec['stem']}_hn{tx}_{ty}", 0, 0, [], [])   # t_img IS the 512px tile, so (0,0) crops all of it
print(f"hard-negative tiles added: {n_hn} ({time.time()-t0:.0f}s)")
mB = YOLO(str(det_A_path)); t0 = time.time()
mB.train(data=str((TILES_DIR / "data.yaml").resolve()), epochs=DET_B_EPOCHS, time=DET_B_HOURS, batch=DET_BATCH, imgsz=DET_IMGSZ, device=DET_DEVICES,
         project=str(TEMP_DIR / "runs_v4"), name="detector_B", exist_ok=True, workers=4, seed=SEED, deterministic=False,
         optimizer="AdamW", lr0=2e-4, lrf=0.1, cos_lr=True, warmup_epochs=1.0, mosaic=0.5, close_mosaic=4, scale=0.3, flipud=0.5, fliplr=0.5,
         degrees=0.0, mixup=0.0, erasing=0.0, amp=True, patience=6, save=True, plots=False, verbose=False)
det_final_path = WORKING_DIR / "yolov8_detector_best.pt"
shutil.copy(Path(mB.trainer.save_dir) / "weights" / "best.pt", det_final_path)
detector = YOLO(str(det_final_path)); yolo_train_min = None
print(f"Stage 1B done in {(time.time()-t0)/60:.1f} min -> {det_final_path}")

hard-negative tiles added: 29 (507s)
Ultralytics 8.4.218 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=4, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/tmp/tiles_v4/data.yaml, degrees=0.0, deterministic=False, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.0, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.0002, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/tmp

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
        2/6         7G      1.536      1.793      1.577          4        640: 100% ━━━━━━━━━━━━ 741/741 2.3it/s 5:24
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 60/60 2.8it/s 21.1s
                   all       3792        546      0.596      0.583      0.585      0.307
Closing dataloader mosaic
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
        3/6      7.01G      1.428      1.633      1.553          1        640: 100% ━━━━━━━━━━━━ 741/741 2.3it/s 5:24
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 60/60 2.8it/s 21.4s
                   al

In [10]:
# Step 7: honest detector evaluation on held-out Zenodo val/test images (natural background mix; thresholds chosen on VAL only)
from torchvision.ops import box_iou, nms as tv_nms
def collect_tiles(recs, n_def, n_clean, seed, per_img=16):
    """Sample images, predict permissively ONCE; tiles are weighted by how many tiles of their kind they represent (unbiased P/R)."""
    rng = random.Random(seed); pool = [r for r in recs if any(b[0] == "defect" for b in r["boxes"])]; ps = {id(x) for x in pool}; clean = [r for r in recs if id(r) not in ps]
    rng.shuffle(pool); rng.shuffle(clean); sel = pool[:n_def] + clean[:n_clean]; data = []
    for n_done, rec in enumerate(sel):
        if n_done % 50 == 0: print(f"    {n_done}/{len(sel)}", flush=True)
        with Image.open(rec["path"]) as im0: im = im0.convert("RGB")
        W, H = im.size; grid = slice_grid(W, H); metas = [tile_gt(rec["boxes"], tx, ty) for tx, ty in grid]
        has = [i for i, m in enumerate(metas) if m[0] or m[1]]; hs = set(has); rest = [i for i in range(len(grid)) if i not in hs]
        rng.shuffle(has); rng.shuffle(rest); k_has = has[:per_img // 2]; k_rest = rest[:per_img - len(k_has)]
        w_has = len(has) / max(len(k_has), 1); w_rest = len(rest) / max(len(k_rest), 1)
        keep = [(i, w_has) for i in k_has] + [(i, w_rest) for i in k_rest]
        tiles = [im.crop((grid[i][0], grid[i][1], grid[i][0] + TILE_SIZE, grid[i][1] + TILE_SIZE)) for i, _ in keep]
        for s in range(0, len(tiles), 32):
            res = detector.predict(tiles[s:s+32], imgsz=DET_IMGSZ, conf=0.05, iou=0.8, classes=[0], verbose=False, device=PRED_DEVICE)
            for (gi, w), r in zip(keep[s:s+32], res):
                gd = metas[gi][0]; b = r.boxes.xyxy.cpu() if len(r.boxes) else torch.zeros((0, 4)); c = r.boxes.conf.cpu() if len(r.boxes) else torch.zeros(0)
                M = box_iou(b, torch.tensor(gd, dtype=torch.float32)).numpy() if (len(b) and len(gd)) else np.zeros((len(b), len(gd)))
                data.append(dict(c=c.numpy(), M=M, b=b, ng=len(gd), w=w, nms={}))
    return data

def kept_after_nms(t, nms_iou):
    if nms_iou not in t["nms"]:
        t["nms"][nms_iou] = tv_nms(t["b"], torch.from_numpy(t["c"]), nms_iou).numpy() if len(t["c"]) else np.zeros(0, int)
    return t["nms"][nms_iou]

def eval_point(tiles, nms_iou, conf, match_iou=0.5):
    tp = npred = gt = 0.0
    for t in tiles:
        gt += t["ng"] * t["w"]; k = kept_after_nms(t, nms_iou); k = k[t["c"][k] >= conf]; npred += len(k) * t["w"]
        if len(k) and t["ng"]:
            hit = np.zeros(t["ng"], bool)
            for i in k[np.argsort(-t["c"][k])]:
                row = np.where(hit, -1.0, t["M"][i]); j = int(row.argmax())
                if row[j] >= match_iou: hit[j] = True; tp += t["w"]
    P, R = tp / max(npred, 1e-9), tp / max(gt, 1e-9)
    return P, R, 2 * P * R / max(P + R, 1e-9)

print("collecting VAL tiles ..."); val_tiles = collect_tiles(val_recs, 120, 60, SEED)
best_op, best_f = None, -1
for n_ in NMS_GRID:
    for c in CONF_GRID:
        f = eval_point(val_tiles, n_, c)[2]
        if f > best_f: best_op, best_f = (n_, c), f
NMS_IOU, DET_CONF = best_op
print(f"operating point chosen on VAL: nms_iou={NMS_IOU}, conf={DET_CONF} (val F1 {best_f:.3f})")
print("collecting TEST tiles ..."); test_tiles = collect_tiles(test_recs, 120, 60, SEED + 1)
det_tile = {}
for thr in (0.5, 0.3):
    P, R, F1 = eval_point(test_tiles, NMS_IOU, DET_CONF, thr); det_tile[thr] = dict(P=P, R=R, F1=F1)
    print(f"TEST tile-level @IoU {thr}: precision {P:.3f} | recall {R:.3f} | F1 {F1:.3f}")
P, R, _ = eval_point(test_tiles, NMS_IOU, 0.05, 0.5); det_tile["ceiling"] = dict(P=P, R=R)
print(f"ceiling (conf 0.05): recall {R:.3f} at precision {P:.3f}")

# AP50 on a natural tile mix (ALL tiles of the sampled images, so ~95% background like deployment) - comparable to a standard mAP50
def build_natural_eval(recs, name, n_def, n_clean, seed):
    d = TEMP_DIR / "ap_eval" / name; shutil.rmtree(d, ignore_errors=True); rng = random.Random(seed)
    pool = [r for r in recs if any(b[0] == "defect" for b in r["boxes"])]; ps = {id(x) for x in pool}; clean = [r for r in recs if id(r) not in ps]; rng.shuffle(pool); rng.shuffle(clean)
    for rec in pool[:n_def] + clean[:n_clean]:
        with Image.open(rec["path"]) as im0: im = im0.convert("RGB")
        W, H = im.size
        for tx, ty in slice_grid(W, H):
            dd, pp = tile_gt(rec["boxes"], tx, ty); write_tile(im, d, rec["stem"], tx, ty, dd, pp)
    y = d / "data.yaml"; y.write_text(yaml.safe_dump({"path": str(d), "train": "images", "val": "images", "names": DET_NAMES})); return y
ap_res = {}
for name, recs in (("val", val_recs), ("test", test_recs)):
    y = build_natural_eval(recs, name, AP_EVAL_DEF_IMGS, AP_EVAL_CLEAN_IMGS, SEED)
    m = YOLO(str(det_final_path)).val(data=str(y), split="val", imgsz=DET_IMGSZ, batch=16, device=PRED_DEVICE, conf=0.001, iou=0.6, plots=False, verbose=False)
    idx = list(m.box.ap_class_index).index(0); p_, r_, ap50, ap = m.box.class_result(idx)
    ap_res[name] = dict(ap50=float(ap50), ap50_95=float(ap), P=float(p_), R=float(r_))
    print(f"[{name}] natural-mix AP50 {ap50:.4f} | AP50-95 {ap:.4f} | P {p_:.4f} | R {r_:.4f}")

collecting VAL tiles ...
    0/148
    50/148
    100/148
operating point chosen on VAL: nms_iou=0.4, conf=0.3 (val F1 0.553)
collecting TEST tiles ...
    0/159
    50/159
    100/159
    150/159
TEST tile-level @IoU 0.5: precision 0.552 | recall 0.508 | F1 0.530
TEST tile-level @IoU 0.3: precision 0.641 | recall 0.590 | F1 0.614
ceiling (conf 0.05): recall 0.781 at precision 0.223
Ultralytics 8.4.218 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 92 layers, 25,840,918 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1327.7±349.6 MB/s, size: 48.8 KB)
val: Scanning /tmp/ap_eval/val/labels... 22708 images, 22432 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 22708/22708 1.8Kit/s 13.0s
val: New cache created: /tmp/ap_eval/val/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1420/1420 2.5it/s 9:37
                   all      22708        286     

In [11]:
# Step 8: classifier crops (Zenodo = evaluated; wafer = auxiliary)
for c in STAGE2_CLASSES:
    (CROPS_DIR / "zenodo" / c).mkdir(parents=True, exist_ok=True); (CROPS_DIR / "wafer" / c).mkdir(parents=True, exist_ok=True)
crop_records = []
def add_crop(im_crop, domain, label, name, group, source):
    out = CROPS_DIR / domain / label / f"{name}.jpg"; im_crop.save(out, quality=95, subsampling=0)
    crop_records.append(dict(path=out, label=label, group=group, domain=domain, source=source))

sizes = [int(crop_side(b[4] - b[2], b[5] - b[3], r["W"], r["H"])) for r in zen_records for b in r["boxes"] if b[0] == "defect" and b[1] in DEFECT_CLASSES and b[4] > b[2] and b[5] > b[3]]
rng = random.Random(SEED); n_particle = n_bg = 0; t0 = time.time()
zr = list(zen_records); rng.shuffle(zr)
for i, rec in enumerate(zr):
    with Image.open(rec["path"]) as im0: im = im0.convert("RGB")
    W, H = im.size; parts = []
    for j, (role, label, x0, y0, x1, y1) in enumerate(rec["boxes"]):
        if x1 - x0 < 1 or y1 - y0 < 1: continue
        if role == "defect" and label in DEFECT_CLASSES: add_crop(context_crop(im, (x0, y0, x1, y1)), "zenodo", label, f"{rec['stem']}_b{j}", rec["group"], "gt")
        elif role == "particle": parts.append((j, (x0, y0, x1, y1)))
    for j, b in rng.sample(parts, min(PARTICLE_PER_IMAGE, len(parts))):
        if n_particle < PARTICLE_CROP_CAP: n_particle += 1; add_crop(context_crop(im, b), "zenodo", "no_defect", f"{rec['stem']}_p{j}", rec["group"], "particle")
    if n_bg < BG_CROP_CAP and sizes:
        cand = [(tx, ty) for tx, ty in slice_grid(W, H) if not touches_any(rec["boxes"], tx, ty)]
        if cand:
            tx, ty = rng.choice(cand); side = int(min(max(rng.choice(sizes), 32), W, H, TILE_SIZE))
            x_hi, y_hi = min(tx + TILE_SIZE, W) - side, min(ty + TILE_SIZE, H) - side
            if x_hi >= tx and y_hi >= ty:
                ox, oy = rng.randint(tx, x_hi), rng.randint(ty, y_hi); n_bg += 1
                add_crop(im.crop((ox, oy, ox + side, oy + side)), "zenodo", "no_defect", f"{rec['stem']}_bg", rec["group"], "empty_tile")
    if (i + 1) % 1000 == 0: print(f"  zenodo crops {i+1}/{len(zr)} ({time.time()-t0:.0f}s)", flush=True)

n_hn_c = 0   # hard negatives from the FINAL detector, on ANY Zenodo image (each crop keeps its image's group, so CV folds stay clean)
for rec, tx, ty, t_img, bxs in mine_fp_tiles(detector, zen_records, HN_CROP_IMAGES, 6, HN_CONF, SEED + 7):
    for k, b in enumerate(bxs):
        if n_hn_c < HN_CROP_CAP:
            n_hn_c += 1; add_crop(context_crop(t_img, b), "zenodo", "no_defect", f"{rec['stem']}_hn{tx}_{ty}_{k}", rec["group"], "hard_negative")
if wafer_records:
    per_cls = defaultdict(list)
    for rec in wafer_records:
        for j, (role, label, x0, y0, x1, y1) in enumerate(rec["boxes"]):
            if x1 - x0 >= 1 and y1 - y0 >= 1: per_cls[label].append((rec, j))
    for label, items in per_cls.items():
        random.Random(SEED).shuffle(items)
        for rec, j in items[:WAFER_CROP_CAP_PER_CLASS]:
            with Image.open(rec["path"]) as im0: im = im0.convert("RGB")
            _, _, x0, y0, x1, y1 = rec["boxes"][j]; add_crop(context_crop(im, (x0, y0, x1, y1)), "wafer", label, f"{rec['stem']}_b{j}", rec["group"], "gt")
import pandas as pd
crop_df = pd.DataFrame([{k: (str(v) if k == "path" else v) for k, v in r.items()} for r in crop_records]); crop_df.to_csv(WORKING_DIR / "crop_manifest.csv", index=False)
print(crop_df.pivot_table(index="label", columns="domain", values="path", aggfunc="count", fill_value=0))
print("Zenodo no_defect sources:", dict(Counter(r["source"] for r in crop_records if r["domain"] == "zenodo" and r["label"] == "no_defect")))

  zenodo crops 1000/4929 (124s)
  zenodo crops 2000/4929 (248s)
  zenodo crops 3000/4929 (373s)
  zenodo crops 4000/4929 (496s)
domain         wafer  zenodo
label                       
contamination   1500      44
inclusion          0     521
no_defect          0    3802
pinhole          251      19
scratch          134      84
Zenodo no_defect sources: {'empty_tile': 2500, 'particle': 1273, 'hard_negative': 29}


In [12]:
# Step 9: classifier - grouped K-fold cross-validation (every Zenodo crop is predicted by a model that never saw its image/specimen group)
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
import torchvision.transforms as TF
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, f1_score
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
class RandomRot90:
    def __call__(self, img):
        k = random.randint(0, 3); return img.rotate(90 * k, expand=True) if k else img
train_tf = TF.Compose([TF.RandomResizedCrop(CLS_IMG_SIZE, scale=(0.7, 1.0), ratio=(0.85, 1.18)), TF.RandomHorizontalFlip(), TF.RandomVerticalFlip(), RandomRot90(),
    TF.RandomApply([TF.ColorJitter(brightness=0.2, contrast=0.2)], p=0.5), TF.RandomApply([TF.GaussianBlur(5, (0.1, 1.5))], p=0.2), TF.ToTensor(), TF.Normalize(MEAN, STD)])
EVAL_TF = TF.Compose([TF.Resize((CLS_IMG_SIZE, CLS_IMG_SIZE)), TF.ToTensor(), TF.Normalize(MEAN, STD)])
class CropDataset(Dataset):
    def __init__(self, recs, tf): self.recs, self.tf = recs, tf
    def __len__(self): return len(self.recs)
    def __getitem__(self, i):
        r = self.recs[i]
        with Image.open(r["path"]) as im: img = im.convert("RGB")
        return self.tf(img), c2i[r["label"]]
class MixedBalancedSampler(Sampler):
    """Per epoch: EPOCH_CAP draws per class. Classes that exist in both domains take 50% Zenodo / 50% wafer, so wafer can never swamp Zenodo."""
    def __init__(self, recs, cap, seed):
        self.cap, self.seed, self.epoch = cap, seed, 0; self.groups = []
        for c in STAGE2_CLASSES:
            z = np.array([i for i, r in enumerate(recs) if r["label"] == c and r["domain"] == "zenodo"], dtype=int)
            w = np.array([i for i, r in enumerate(recs) if r["label"] == c and r["domain"] == "wafer"], dtype=int)
            if len(z) or len(w): self.groups.append((z, w))
    def __iter__(self):
        rng = np.random.default_rng(self.seed + self.epoch); self.epoch += 1; idx = []
        for z, w in self.groups:
            if len(z) and len(w):
                nz = self.cap // 2; nw = self.cap - nz
                idx += list(rng.choice(z, nz, replace=len(z) < nz)) + list(rng.choice(w, nw, replace=len(w) < nw))
            else:
                pool = z if len(z) else w; idx += list(rng.choice(pool, self.cap, replace=len(pool) < self.cap))
        rng.shuffle(idx); return iter([int(i) for i in idx])
    def __len__(self): return self.cap * len(self.groups)

class AdaptiveConcatPool2d(nn.Module):
    def __init__(self): super().__init__(); self.avg, self.max = nn.AdaptiveAvgPool2d(1), nn.AdaptiveMaxPool2d(1)
    def forward(self, x): return torch.cat([self.avg(x), self.max(x)], 1)
def build_classifier(name=CLS_BACKBONE, num_classes=len(STAGE2_CLASSES), pretrained=True):
    base = getattr(_tvm, name)(weights="DEFAULT" if pretrained else None); feat = base.fc.in_features * 2
    return nn.Sequential(nn.Sequential(*list(base.children())[:-2]),
        nn.Sequential(AdaptiveConcatPool2d(), nn.Flatten(), nn.BatchNorm1d(feat), nn.Dropout(0.3), nn.Linear(feat, 256), nn.ReLU(inplace=True), nn.BatchNorm1d(256), nn.Dropout(0.2), nn.Linear(256, num_classes)))
class EMA:
    def __init__(self, model, decay):
        self.module = copy.deepcopy(model).eval(); self.decay, self.n = decay, 0
        for p in self.module.parameters(): p.requires_grad_(False)
    @torch.no_grad()
    def update(self, model):
        self.n += 1; d = min(self.decay, (1 + self.n) / (10 + self.n)); msd = model.state_dict()
        for k, v in self.module.state_dict().items():
            if v.dtype.is_floating_point: v.mul_(d).add_(msd[k].detach(), alpha=1 - d)
            else: v.copy_(msd[k])

device = torch.device("cuda:0" if cuda_avail else "cpu"); use_amp = device.type == "cuda"; n_workers = min(4, os.cpu_count() or 1)
def _loader(recs, tf, **kw): return DataLoader(CropDataset(recs, tf), batch_size=CLS_BATCH, num_workers=n_workers, pin_memory=True, **kw)

@torch.no_grad()
def predict_logits(m, recs):
    m.eval(); outs = []
    for x, _ in _loader(recs, EVAL_TF):
        with torch.autocast(device_type=device.type, enabled=use_amp): lg = m(x.to(device, non_blocking=True))
        outs.append(lg.float().cpu())
    return torch.cat(outs) if outs else torch.zeros((0, len(STAGE2_CLASSES)))

def macro_f1_present(y, pred):
    labs = np.unique(y); return float(f1_score(y, pred, labels=labs, average="macro", zero_division=0)) if len(labs) else 0.0

def train_classifier(train_r, val_r, seed):
    model = build_classifier().to(device); loss_fn = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH); scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    sampler = MixedBalancedSampler(train_r, EPOCH_CAP, seed); loader = _loader(train_r, train_tf, sampler=sampler, drop_last=True)
    yv = np.array([c2i[r["label"]] for r in val_r])
    def run_epoch(opt, freeze_bn, ema=None):
        model.train()
        if freeze_bn:
            for m in model[0].modules():
                if isinstance(m, nn.BatchNorm2d): m.eval()
        params = [p for g in opt.param_groups for p in g["params"]]
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True); opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=use_amp): loss = loss_fn(model(x), y)
            scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0); scaler.step(opt); scaler.update()
            if ema is not None: ema.update(model)
    for p in model[0].parameters(): p.requires_grad = False
    opt = torch.optim.AdamW(model[1].parameters(), lr=1e-3, weight_decay=1e-3)
    for _ in range(CLS_WARMUP_EPOCHS): run_epoch(opt, True)
    for p in model[0].parameters(): p.requires_grad = True
    opt = torch.optim.AdamW([{"params": model[0].parameters(), "lr": 5e-5, "weight_decay": 1e-4}, {"params": model[1].parameters(), "lr": 5e-4, "weight_decay": 1e-3}])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CLS_EPOCHS, eta_min=1e-6); ema = EMA(model, EMA_DECAY)
    best_f1, best_state, bad = -1.0, None, 0
    for ep in range(1, CLS_EPOCHS + 1):
        run_epoch(opt, False, ema); sched.step()
        f1 = macro_f1_present(yv, predict_logits(ema.module, val_r).argmax(1).numpy())
        if f1 > best_f1 + 1e-4: best_f1, bad = f1, 0; best_state = {k: v.detach().cpu().clone() for k, v in ema.module.state_dict().items()}
        else: bad += 1
        if bad >= CLS_PATIENCE: break
    return best_state, best_f1, ep

z_recs = [r for r in crop_records if r["domain"] == "zenodo"]; w_recs = [r for r in crop_records if r["domain"] == "wafer"]
fold_of_group = assign_folds([(r["group"], r["strat"]) for r in zen_records])      # ALL images, so every group has a fold (also those without crops)
for r in z_recs: r["fold"] = fold_of_group[r["group"]]
print("Zenodo crops per fold:", {f: sum(r["fold"] == f for r in z_recs) for f in range(K_FOLDS)})
print("defect crops per fold:", {f: dict(Counter(r["label"] for r in z_recs if r["fold"] == f and r["label"] != "no_defect")) for f in range(K_FOLDS)})
oof = {}; fold_states = []; t0 = time.time()
for f in range(K_FOLDS):
    test_r = [r for r in z_recs if r["fold"] == f]; val_r = [r for r in z_recs if r["fold"] == (f + 1) % K_FOLDS]
    train_r = [r for r in z_recs if r["fold"] not in (f, (f + 1) % K_FOLDS)] + w_recs
    state, bf, ep = train_classifier(train_r, val_r, SEED + f); fold_states.append(state)
    m = build_classifier(pretrained=False).to(device); m.load_state_dict(state)
    lg = predict_logits(m, test_r)
    for r, l in zip(test_r, lg): oof[id(r)] = l.numpy()
    acc = float((lg.argmax(1).numpy() == np.array([c2i[r["label"]] for r in test_r])).mean())
    print(f"fold {f}: train {len(train_r)} | val macro-F1 {bf:.3f} ({ep} ep) | held-out acc {acc*100:.1f}% (n={len(test_r)}) | {(time.time()-t0)/60:.1f} min", flush=True)
    del m; torch.cuda.empty_cache()

Zenodo crops per fold: {0: 946, 1: 956, 2: 766, 3: 1033, 4: 769}
defect crops per fold: {0: {'scratch': 14, 'inclusion': 106, 'contamination': 7, 'pinhole': 4}, 1: {'contamination': 8, 'inclusion': 103, 'scratch': 26, 'pinhole': 4}, 2: {'inclusion': 105, 'pinhole': 4, 'scratch': 9, 'contamination': 6}, 3: {'inclusion': 101, 'scratch': 22, 'contamination': 15, 'pinhole': 3}, 4: {'inclusion': 106, 'scratch': 13, 'pinhole': 4, 'contamination': 8}}
Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 216MB/s]


fold 0: train 4453 | val macro-F1 0.632 (10 ep) | held-out acc 93.7% (n=946) | 3.0 min
fold 1: train 4633 | val macro-F1 0.728 (15 ep) | held-out acc 95.7% (n=956) | 7.3 min
fold 2: train 4556 | val macro-F1 0.751 (19 ep) | held-out acc 95.0% (n=766) | 12.7 min
fold 3: train 4553 | val macro-F1 0.773 (18 ep) | held-out acc 96.8% (n=1033) | 17.7 min
fold 4: train 4640 | val macro-F1 0.724 (25 ep) | held-out acc 95.4% (n=769) | 24.7 min


In [13]:
# Step 10: out-of-fold metrics + temperature + confidence intervals
oof_recs = [r for r in z_recs if id(r) in oof]
L = torch.tensor(np.stack([oof[id(r)] for r in oof_recs])); Y = np.array([c2i[r["label"]] for r in oof_recs]); G = np.array([r["group"] for r in oof_recs])
log_T = torch.zeros(1, requires_grad=True); lb = torch.optim.LBFGS([log_T], lr=0.1, max_iter=100); Yt = torch.tensor(Y)
def _cl():
    lb.zero_grad(); l = F.cross_entropy(L / log_T.exp(), Yt); l.backward(); return l
lb.step(_cl); TEMP = float(log_T.exp().item()); print(f"Temperature (fit on out-of-fold logits) T = {TEMP:.3f}")
PR = F.softmax(L / TEMP, 1).numpy(); PRED = PR.argmax(1); n = len(Y)
acc = float((PRED == Y).mean()); mf1 = macro_f1_present(Y, PRED); dmask = Y != c2i["no_defect"]; dacc = float((PRED[dmask] == Y[dmask]).mean())
def group_bootstrap(y, pred, groups, n_boot=300, seed=SEED):
    rng = np.random.default_rng(seed); ug = np.unique(groups); idx_by = {u: np.where(groups == u)[0] for u in ug}; a, f = [], []
    for _ in range(n_boot):
        ids = np.concatenate([idx_by[u] for u in rng.choice(ug, len(ug), replace=True)]); a.append((pred[ids] == y[ids]).mean()); f.append(macro_f1_present(y[ids], pred[ids]))
    return np.percentile(a, [2.5, 97.5]), np.percentile(f, [2.5, 97.5])
acc_ci, f1_ci = group_bootstrap(Y, PRED, G)
print(f"OUT-OF-FOLD (n={n} Zenodo crops): accuracy {acc*100:.2f}% (95% CI {acc_ci[0]*100:.2f}-{acc_ci[1]*100:.2f}) | macro-F1 {mf1:.4f} (95% CI {f1_ci[0]:.4f}-{f1_ci[1]:.4f})")
print(f"defect-classes-only accuracy {dacc*100:.2f}% (n={int(dmask.sum())})")
C = len(STAGE2_CLASSES); conf = confusion_matrix(Y, PRED, labels=list(range(C)))
p_, r_, f_, s_ = precision_recall_fscore_support(Y, PRED, labels=list(range(C)), zero_division=0)
print(f"\n{'class':<15}|{'P':>8}|{'R':>8}|{'F1':>8}|{'n':>6}")
for i, c in enumerate(STAGE2_CLASSES): print(f"{c:<15}|{p_[i]*100:7.1f}%|{r_[i]*100:7.1f}%|{f_[i]*100:7.1f}%|{int(s_[i]):>6d}" + ("   LOW-DATA (n<30): interpret with caution" if s_[i] < 30 else ""))
print("\nConfusion (rows=true, cols=pred):"); print(" " * 15 + "".join(f"{c[:7]:>9}" for c in STAGE2_CLASSES))
for i, c in enumerate(STAGE2_CLASSES): print(f"{c:<15}" + "".join(f"{conf[i, j]:>9}" for j in range(C)))
src = np.array([r["source"] for r in oof_recs])
for s in sorted(set(src[Y == c2i["no_defect"]])): m = (Y == c2i["no_defect"]) & (src == s); print(f"no_defect recall [{s}]: {(PRED[m] == c2i['no_defect']).mean()*100:.1f}% (n={int(m.sum())})")
# confident disagreements, for a HUMAN label review (never auto-delete)
sus = np.where((PRED != Y) & (PR.max(1) >= 0.9))[0]; sus = sus[np.argsort(-PR[sus].max(1))]
pd.DataFrame([dict(path=str(oof_recs[i]["path"]), label=oof_recs[i]["label"], predicted=i2c[PRED[i]], p=round(float(PR[i].max()), 3)) for i in sus]).to_csv(WORKING_DIR / "label_audit_suspects.csv", index=False)
print(f"\nlabel-audit: {len(sus)} confident disagreements -> label_audit_suspects.csv")
cls_bundle = WORKING_DIR / "resnet_classifier_cv.pt"
torch.save(dict(arch=CLS_BACKBONE, classes=STAGE2_CLASSES, img_size=CLS_IMG_SIZE, mean=MEAN, std=STD, temperature=TEMP, fold_states=fold_states, fold_of_group=fold_of_group, k=K_FOLDS,
                oof=dict(accuracy=acc, macro_f1=mf1)), cls_bundle)
fold_models = []
for st in fold_states:
    m = build_classifier(pretrained=False).to(device); m.load_state_dict(st); m.eval(); fold_models.append(m)

Temperature (fit on out-of-fold logits) T = 0.877
OUT-OF-FOLD (n=4470 Zenodo crops): accuracy 95.37% (95% CI 94.69-95.97) | macro-F1 0.6928 (95% CI 0.6256-0.7387)
defect-classes-only accuracy 85.78% (n=668)

class          |       P|       R|      F1|     n
contamination  |   57.7%|   68.2%|   62.5%|    44
inclusion      |   81.0%|   91.9%|   86.2%|   521
no_defect      |   99.2%|   97.1%|   98.1%|  3802
pinhole        |   35.3%|   31.6%|   33.3%|    19   LOW-DATA (n<30): interpret with caution
scratch        |   63.7%|   69.0%|   66.3%|    84

Confusion (rows=true, cols=pred):
                 contami  inclusi  no_defe  pinhole  scratch
contamination         30        4        5        0        5
inclusion              2      479       19        8       13
no_defect             17       78     3690        3       14
pinhole                0       12        0        6        1
scratch                3       18        5        0       58
no_defect recall [empty_tile]: 99.6% (n=2500)
no_

In [14]:
# Step 11: end-to-end on held-out Zenodo images. Each image is classified ONLY by the fold model that never saw its group.
import torchvision
nd = c2i["no_defect"]
@torch.no_grad()
def classify_crops(crops, fold):
    out = []
    for i in range(0, len(crops), 64):
        xb = torch.stack([EVAL_TF(c) for c in crops[i:i+64]]).to(device)
        with torch.autocast(device_type=device.type, enabled=use_amp): lg = fold_models[fold](xb)
        out.append(F.softmax(lg.float() / TEMP, 1).cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, len(STAGE2_CLASSES)))

def detect_image(im, conf=0.05):
    W, H = im.size; grid = slice_grid(W, H); tiles = [im.crop((x, y, x + TILE_SIZE, y + TILE_SIZE)) for x, y in grid]; bx, cf = [], []
    for i in range(0, len(tiles), 32):
        res = detector.predict(tiles[i:i+32], imgsz=DET_IMGSZ, conf=conf, iou=0.8, classes=[0], verbose=False, device=PRED_DEVICE)
        for (ox, oy), r in zip(grid[i:i+32], res):
            if len(r.boxes):
                b = r.boxes.xyxy.cpu().numpy().copy(); b[:, [0, 2]] += ox; b[:, [1, 3]] += oy; b[:, [0, 2]] = b[:, [0, 2]].clip(0, W); b[:, [1, 3]] = b[:, [1, 3]].clip(0, H)
                ok = (b[:, 2] - b[:, 0] > 1) & (b[:, 3] - b[:, 1] > 1); bx.append(b[ok]); cf.append(r.boxes.conf.cpu().numpy()[ok])
    if not bx: return np.zeros((0, 4)), np.zeros(0)
    b, c = np.vstack(bx), np.concatenate(cf)
    if len(b) == 0: return np.zeros((0, 4)), np.zeros(0)
    k = torchvision.ops.nms(torch.from_numpy(b).float(), torch.from_numpy(c).float(), NMS_IOU).numpy(); return b[k], c[k]

def e2e_cache(recs):
    cache = {}
    for i, rec in enumerate(recs):
        with Image.open(rec["path"]) as im0: im = im0.convert("RGB")
        b, c = detect_image(im); order = np.argsort(-c)[:150]; b, c = b[order], c[order]
        probs = classify_crops([context_crop(im, x) for x in b], fold_of_group[rec["group"]]) if len(b) else np.zeros((0, len(STAGE2_CLASSES)))
        cache[rec["stem"]] = [dict(box=[float(v) for v in bb], det_conf=float(cc), probs=pp) for bb, cc, pp in zip(b, c, probs)]
        if (i + 1) % 50 == 0: print(f"    {i+1}/{len(recs)}", flush=True)
    return cache

def filter_dets(raw, conf_thr, rej_thr):
    out = []
    for d in raw:
        if d["det_conf"] < conf_thr: continue
        p = d["probs"]; sub = p.copy(); sub[nd] = -1; k = int(sub.argmax())
        if p[nd] >= rej_thr and p[nd] > p[k]: continue
        out.append(dict(box=d["box"], conf=d["det_conf"], subtype=i2c[k]))
    return out

def score_e2e(recs, cache, conf_thr, rej_thr, iou_thr, prevalence):
    tp = npred = gt = sub_ok = lab_tp = fp_clean = n_clean = n_def = fp_def = 0; per_cls = defaultdict(Counter)
    for r in recs:
        dets = sorted(filter_dets(cache[r["stem"]], conf_thr, rej_thr), key=lambda d: -d["conf"]); gts = [(b[1], b[2:]) for b in r["boxes"] if b[0] == "defect"]
        if not gts: n_clean += 1; fp_clean += len(dets); continue
        n_def += 1; gt += len(gts); used = set(); t_here = 0
        for lab, _ in gts:
            if lab: per_cls[lab]["gt"] += 1
        for d in dets:
            bj, bi = -1, iou_thr
            for j, (lab, box) in enumerate(gts):
                if j not in used:
                    v = iou_xyxy(d["box"], box)
                    if v >= bi: bj, bi = j, v
            if bj >= 0:
                used.add(bj); tp += 1; t_here += 1; lab = gts[bj][0]
                if lab: lab_tp += 1; ok = d["subtype"] == lab; sub_ok += ok; per_cls[lab]["hit"] += 1; per_cls[lab]["sub_ok"] += ok
        fp_def += len(dets) - t_here; npred += len(dets)
    rec_ = tp / max(gt, 1); fp_c = fp_clean / max(n_clean, 1); tp_d = tp / max(n_def, 1); fp_d = fp_def / max(n_def, 1)
    denom = prevalence * (tp_d + fp_d) + (1 - prevalence) * fp_c; p_nat = prevalence * tp_d / denom if denom > 0 else float("nan")
    return dict(recall=rec_, precision_natural=p_nat, f1=2 * rec_ * p_nat / max(rec_ + p_nat, 1e-9) if p_nat == p_nat else 0.0, subtype_acc=sub_ok / max(lab_tp, 1),
                fp_per_clean_img=fp_c, fp_per_defect_img=fp_d, n_def=n_def, n_clean=n_clean, gt=gt,
                per_class={c: dict(gt=v["gt"], recall=v["hit"] / max(v["gt"], 1), sub_acc=v["sub_ok"] / max(v["hit"], 1)) for c, v in per_cls.items()})

def pick_imgs(recs, nd_, nc_, seed):
    rng = random.Random(seed); d = [r for r in recs if any(b[0] == "defect" for b in r["boxes"])]; ds = {id(x) for x in d}; c = [r for r in recs if id(r) not in ds]; rng.shuffle(d); rng.shuffle(c); return d[:nd_] + c[:nc_]
prev_test = sum(any(b[0] == "defect" for b in r["boxes"]) for r in test_recs) / max(len(test_recs), 1)
prev_val = sum(any(b[0] == "defect" for b in r["boxes"]) for r in val_recs) / max(len(val_recs), 1)
print("caching VAL ..."); v_sel = pick_imgs(val_recs, E2E_DEF_IMGS, E2E_CLEAN_IMGS, SEED); v_cache = e2e_cache(v_sel)
best_e, best_ef = None, -1
for ct in (0.15, 0.2, 0.25, 0.3, 0.4, 0.5):
    for rj in (0.5, 0.7, 0.9, 1.01):
        f = score_e2e(v_sel, v_cache, ct, rj, 0.5, prev_val)["f1"]
        if f > best_ef: best_e, best_ef = (ct, rj), f
E2E_CONF, E2E_REJ = best_e; print(f"end-to-end operating point chosen on VAL: det_conf>={E2E_CONF}, reject if p(no_defect)>={E2E_REJ} (F1 {best_ef:.3f})")
print("caching TEST ..."); t_sel = pick_imgs(test_recs, E2E_DEF_IMGS, E2E_CLEAN_IMGS, SEED + 1); t_cache = e2e_cache(t_sel)
e2e = {}
for thr in (0.5, 0.3):
    s = score_e2e(t_sel, t_cache, E2E_CONF, E2E_REJ, thr, prev_test); e2e[thr] = s
    print(f"\nTEST end-to-end @ IoU {thr} (defect imgs {s['n_def']}, clean imgs {s['n_clean']}, GT boxes {s['gt']}, natural prevalence {prev_test*100:.0f}%)")
    print(f"  recall {s['recall']*100:.1f}% | precision at natural prevalence {s['precision_natural']*100:.1f}% | subtype accuracy {s['subtype_acc']*100:.1f}% | FP per clean image {s['fp_per_clean_img']:.3f}")
    for c, v in s["per_class"].items(): print(f"    {c:<14} GT {v['gt']:3d} | recall {v['recall']*100:5.1f}% | subtype acc {v['sub_acc']*100:5.1f}%" + ("  LOW-DATA" if v["gt"] < 30 else ""))

caching VAL ...
    50/188
    100/188
    150/188
end-to-end operating point chosen on VAL: det_conf>=0.3, reject if p(no_defect)>=0.9 (F1 0.443)
caching TEST ...
    50/199
    100/199
    150/199

TEST end-to-end @ IoU 0.5 (defect imgs 99, clean imgs 100, GT boxes 125, natural prevalence 11%)
  recall 59.2% | precision at natural prevalence 32.4% | subtype accuracy 90.5% | FP per clean image 0.090
    inclusion      GT  88 | recall  58.0% | subtype acc  98.0%
    scratch        GT  28 | recall  53.6% | subtype acc  80.0%  LOW-DATA
    pinhole        GT   3 | recall  66.7% | subtype acc   0.0%  LOW-DATA
    contamination  GT   6 | recall 100.0% | subtype acc  83.3%  LOW-DATA

TEST end-to-end @ IoU 0.3 (defect imgs 99, clean imgs 100, GT boxes 125, natural prevalence 11%)
  recall 72.8% | precision at natural prevalence 39.9% | subtype accuracy 90.1% | FP per clean image 0.090
    inclusion      GT  88 | recall  73.9% | subtype acc  96.9%
    scratch        GT  28 | recall  60.7% | su

In [15]:
# Step 12: honest report + package (Zenodo numbers only; no target is assumed to be met)
def _ver(pkg):
    try: return md.version(pkg)
    except Exception: return "n/a"
cfg = dict(seed=SEED, classes=STAGE2_CLASSES, det_imgsz=DET_IMGSZ, tile=TILE_SIZE, overlap=TILE_OVERLAP, nms_iou=NMS_IOU, det_conf_tile=DET_CONF,
           e2e_operating_point=dict(det_conf=E2E_CONF, reject_p_no_defect=E2E_REJ, chosen_on="val"), cls_backbone=CLS_BACKBONE, cls_img_size=CLS_IMG_SIZE, k_folds=K_FOLDS,
           temperature=TEMP, wafer_aux=bool(wafer_records), versions={p: _ver(p) for p in ("torch", "torchvision", "ultralytics", "scikit-learn", "numpy")})
(WORKING_DIR / "config.json").write_text(json.dumps(cfg, indent=2))
(WORKING_DIR / "splits.json").write_text(json.dumps(dict(split_of_group=split_of, fold_of_group=fold_of_group)))   # lets later diagnostics reproduce the exact splits
R = ["# Dhatu Rakshana v4 (Zenodo-first) - report", f"Generated {time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())}", "",
     "All numbers are on held-out Zenodo (coating) images. Wafer data was used for training only.", "",
     f"## Detector ({'reused previous weights + ' if reused else ''}stage A + hard-negative stage B)",
     f"* natural-mix AP50 (val / test): {ap_res['val']['ap50']:.4f} / {ap_res['test']['ap50']:.4f} ; AP50-95 {ap_res['val']['ap50_95']:.4f} / {ap_res['test']['ap50_95']:.4f}",
     f"* tile-level test @IoU0.5 (nms {NMS_IOU}, conf {DET_CONF}): precision {det_tile[0.5]['P']:.3f}, recall {det_tile[0.5]['R']:.3f}, F1 {det_tile[0.5]['F1']:.3f} ; @IoU0.3: P {det_tile[0.3]['P']:.3f}, R {det_tile[0.3]['R']:.3f}",
     f"* recall ceiling at conf 0.05: {det_tile['ceiling']['R']:.3f} (precision {det_tile['ceiling']['P']:.3f})", "",
     f"## Classifier ({CLS_BACKBONE}, {K_FOLDS}-fold grouped CV, out-of-fold)",
     f"* accuracy {acc*100:.2f}% (95% CI {acc_ci[0]*100:.2f}-{acc_ci[1]*100:.2f}); macro-F1 {mf1:.4f} (95% CI {f1_ci[0]:.4f}-{f1_ci[1]:.4f}); defect-only accuracy {dacc*100:.2f}%"]
for i, c in enumerate(STAGE2_CLASSES): R.append(f"  * {c}: P {p_[i]*100:.1f}% R {r_[i]*100:.1f}% F1 {f_[i]*100:.1f}% (n={int(s_[i])})" + (" LOW-DATA" if s_[i] < 30 else ""))
R += ["", f"## End-to-end on held-out test images (det_conf>={E2E_CONF}, reject>={E2E_REJ}, chosen on val)"]
for thr, s in e2e.items(): R.append(f"* IoU {thr}: recall {s['recall']*100:.1f}%, precision at natural prevalence {s['precision_natural']*100:.1f}%, subtype acc {s['subtype_acc']*100:.1f}%, FP/clean image {s['fp_per_clean_img']:.3f}")
(WORKING_DIR / "training_report.md").write_text("\n".join(R), encoding="utf-8"); print("\n".join(R))
bundle_zip = WORKING_DIR / "dhatu_v4_bundle.zip"
with zipfile.ZipFile(bundle_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in [det_final_path, cls_bundle, WORKING_DIR / "config.json", WORKING_DIR / "training_report.md", WORKING_DIR / "splits.json", WORKING_DIR / "crop_manifest.csv", WORKING_DIR / "label_audit_suspects.csv"]:
        if f.exists(): zf.write(f, arcname=f.name)
print(f"\nBundle: {bundle_zip} ({bundle_zip.stat().st_size/1024**2:.1f} MB)")

# Dhatu Rakshana v4 (Zenodo-first) - report
Generated 2026-11-24 10:29:39 UTC

All numbers are on held-out Zenodo (coating) images. Wafer data was used for training only.

## Detector (stage A + hard-negative stage B)
* natural-mix AP50 (val / test): 0.4779 / 0.5068 ; AP50-95 0.2179 / 0.2256
* tile-level test @IoU0.5 (nms 0.4, conf 0.3): precision 0.552, recall 0.508, F1 0.530 ; @IoU0.3: P 0.641, R 0.590
* recall ceiling at conf 0.05: 0.781 (precision 0.223)

## Classifier (resnet34, 5-fold grouped CV, out-of-fold)
* accuracy 95.37% (95% CI 94.69-95.97); macro-F1 0.6928 (95% CI 0.6256-0.7387); defect-only accuracy 85.78%
  * contamination: P 57.7% R 68.2% F1 62.5% (n=44)
  * inclusion: P 81.0% R 91.9% F1 86.2% (n=521)
  * no_defect: P 99.2% R 97.1% F1 98.1% (n=3802)
  * pinhole: P 35.3% R 31.6% F1 33.3% (n=19) LOW-DATA
  * scratch: P 63.7% R 69.0% F1 66.3% (n=84)

## End-to-end on held-out test images (det_conf>=0.3, reject>=0.9, chosen on val)
* IoU 0.5: recall 59.2%, precision at nat